In [68]:
import os
import json
import math
import numpy as np
import time

# plot libraries
import matplotlib.pyplot as plt
import seaborn as sns

# progress bar
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

import networkx as nx


$$H^{(l+1)} = \sigma(\hat{D}^{-1/2}\hat{A}\hat{D}^{-1/2}H^{(l)}W^{(l)})$$

In [ ]:
def step(vertex_features, adj_matrix, weights):
    # find the degree of each vertex
    degrees = adj_matrix.sum(dim=-1) # (n, n) -> (n)
    # compute deg(v)^(-1/2) for all vertices v
    sqrt_inv_degrees = np.sqrt(np.power(degrees, -1)) # (n) -> (n)

    # make diagonal (n, n) matrix form degree vector
    sqrt_inv_degrees_matrix = torch.diag(sqrt_inv_degrees)

    # calculate H * W
    vertex_features = vertex_features @ weights # (n, c_in) * (c_in, c_out) -> (n, c_out)
    # D^(-1/2) * H * W
    vertex_features = sqrt_inv_degrees_matrix @ vertex_features # (n, n) * (n, c_out) -> (n, c_out)
    # A * D^(-1/2) * H * W
    vertex_features = adj_matrix @ vertex_features # (n, n) * (n, c_out) -> (n, c_out)
    # D^(-1/2) * A * D^(-1/2) * H * W
    vertex_features = sqrt_inv_degrees_matrix @ vertex_features # (n, n) * (n, c_out) -> (n, c_out)

    return vertex_features # apply sigmoid

In [128]:
device = torch.device("cpu")

In [129]:
def normalized_adjacency_matrix(adj_matrix: torch.Tensor) -> torch.Tensor:
    """
    Normalizes the adjacency matrix by calculating D^(-1/2)AD^(-1/2)

    :param adj_matrix: adjacency matrix of the graph of shape (N, N)
    :return: normalized adjacency matrix
    """
    adj_matrix = adj_matrix + torch.eye(adj_matrix.shape[0]).to(device)

    # find the degree of each vertex
    degrees = adj_matrix.sum(dim=-1) # (n, n) -> (n)
    # compute deg(v)^(-1/2) for all vertices v
    sqrt_inv_degrees = torch.pow(degrees, -0.5) # (n) -> (n)

    # make diagonal (n, n) matrix form degree vector
    sqrt_inv_degrees_matrix = torch.diag(sqrt_inv_degrees)

    # return D^(-1/2) * A * D^(-1/2)
    return sqrt_inv_degrees_matrix @ adj_matrix @ sqrt_inv_degrees_matrix # matrix of shape (n, n)

In [130]:
class GNNLayer(nn.Module):
    def __init__(self, channels_input, channels_output):
        super().__init__()
        self.weights = nn.Linear(channels_input, channels_output, bias=False) # W weight matrix of shape (I, O)
        self.sigmoid = nn.Sigmoid()

    def forward(self, vertex_features, adj_matrix):
        """

        :param vertex_features: tensor of shape (..., N, I) containing vertex input features
        :param adj_matrix: adjacency matrix of the graph of shape (N, N)
        :return: tensor of shape (..., N, O) containing new vertex output features
        """
        vertex_features = normalized_adjacency_matrix(adj_matrix) @ vertex_features
        return self.sigmoid(self.weights(vertex_features))


In [131]:
class GNN(nn.Module):
    def __init__(self, channels_input, channels_output):
        super().__init__()
        self.gnn_layer1 = GNNLayer(channels_input, 8)
        self.relu = nn.ReLU()
        self.gnn_layer2 = GNNLayer(8, channels_output)

    def forward(self, x, adj_matrix):
        h = x.squeeze(-1)
        h = self.gnn_layer1(h, adj_matrix)
        h = self.relu(h)
        return self.gnn_layer2(h, adj_matrix)

In [132]:
def generate_gnn_dataset(num_graphs=5, num_nodes=50, edge_prob=0.1, feature_dim=16):
    x_batch, adj_batch, y_batch = [], [], []

    for _ in range(num_graphs):
        G = nx.erdos_renyi_graph(n=num_nodes, p=edge_prob)

        # adjacency matrix of shape (N, N)
        adj = torch.tensor(nx.to_numpy_array(G), dtype=torch.float32)
        adj_batch.append(adj)

        # inputs are column vector of shape (N, I)
        features = torch.randn(num_nodes, feature_dim, 1, dtype=torch.float32)
        x_batch.append(features)

        # targets are column vector of degrees of shape (N, 1)
        degrees = [deg for _, deg in G.degree()]
        targets = torch.tensor(degrees, dtype=torch.float32).view(-1, 1)
        y_batch.append(targets)

    # stack lists into batched tensors
    return torch.stack(x_batch), torch.stack(adj_batch), torch.stack(y_batch)

In [117]:
x, adj, y = generate_gnn_dataset(num_graphs=10000, num_nodes=50, edge_prob=0.2, feature_dim=8)

print(f"Node Features (x) shape:        {x.shape}")
print(f"Adjacency Matrix (adj) shape:   {adj.shape}")
print(f"Target Degrees (y) shape:       {y.shape}")

Node Features (x) shape:        torch.Size([10000, 50, 8, 1])
Adjacency Matrix (adj) shape:   torch.Size([10000, 50, 50])
Target Degrees (y) shape:       torch.Size([10000, 50, 1])


In [133]:
def train_custom_gnn(x_batch, adj_batch, y_batch, epochs=100, lr=0.01):
    # extract input feature dimension
    feature_dim = x_batch.size(2)

    # initialize the GNN
    model = GNN(channels_input=feature_dim, channels_output=1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        batch_loss = 0

        # iterate through the batch sequentially to pass (N, N) adjacency matrices
        for i in range(x_batch.size(0)):
            x = x_batch[i].to(device)       # vertex features of shape (N, 8, 1)
            adj = adj_batch[i].to(device)   # adjacency matrix of shape (N, N)
            y = y_batch[i].to(device)       # output vertex features of shape (N, 1)

            predictions = model(x, adj)
            loss = criterion(predictions, y)
            batch_loss += loss

        # average loss across the batch
        batch_loss = batch_loss / x_batch.size(0)

        batch_loss.backward()
        optimizer.step()

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {batch_loss.item():.4f}")

    return model

In [116]:
print("Starting training...")
trained_model = train_custom_gnn(x, adj, y, epochs=100)

Starting training...
Epoch  20/100 | Loss: 89.3065
Epoch  40/100 | Loss: 87.5858
Epoch  60/100 | Loss: 86.7532
Epoch  80/100 | Loss: 86.3292
Epoch 100/100 | Loss: 86.0867
